In [1]:
import pandas as pd
import pickle

from sklearn.metrics import (
    roc_auc_score,
    log_loss
)

from sklearn.model_selection import train_test_split

In [2]:
X = pd.read_csv("../ctr_X.csv")
y = pd.read_csv("../ctr_y.csv").squeeze()

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [4]:
with open("../ctr_model.pkl", "rb") as f:
    ctr_model = pickle.load(f)

print("CTR model loaded successfully!")

CTR model loaded successfully!


In [5]:
click_prob = ctr_model.predict_proba(X_test)

click_prob[:5]

array([[0.24558457, 0.75441543],
       [0.60459346, 0.39540654],
       [0.47418251, 0.52581749],
       [0.34390244, 0.65609756],
       [0.36324577, 0.63675423]])

In [6]:
auc = roc_auc_score(
    y_test,
    click_prob[:, 1]
)

print("AUC-ROC:", auc)

AUC-ROC: 0.8460815785996856


In [7]:
loss = log_loss(
    y_test,
    click_prob[:, 1]
)

print("Log Loss:", loss)

Log Loss: 0.4834352041091077


In [8]:
results = X_test.copy()

results["actual"] = y_test.values

results["probability"] = click_prob[:, 1]

results = results.sort_values(
    "probability",
    ascending=False
)

In [9]:
top10 = results.head(10)

top10

,avg_rating,rating_count,avg_movie_rating,movie_rating_count,actual,probability
436801,5.0,116,5.0,1,1,0.997305
381817,5.0,116,5.0,1,1,0.997305
476640,5.0,116,5.0,1,1,0.997305
438369,5.0,116,5.0,1,1,0.997305
86264,5.0,116,5.0,1,1,0.997305
357813,5.0,116,5.0,1,1,0.997305
464681,5.0,116,5.0,1,1,0.997305
400552,5.0,116,5.0,1,1,0.997305
192641,5.0,116,5.0,1,1,0.997305
299182,5.0,116,5.0,1,1,0.997305


In [10]:
ctr_at_10 = top10["actual"].mean()

print("CTR@10:", ctr_at_10)

CTR@10: 1.0


In [11]:
evaluation = pd.DataFrame({
    "Metric": [
        "AUC-ROC",
        "Log Loss",
        "CTR@10"
    ],
    "Value": [
        auc,
        loss,
        ctr_at_10
    ]
})

evaluation.to_csv("../ctr_evaluation.csv", index=False)

evaluation

,Metric,Value
0,AUC-ROC,0.846082
1,Log Loss,0.483435
2,CTR@10,1.000000


# Day 10 — CTR Model Evaluation

## Objective
Evaluate the Logistic Regression CTR prediction model using ranking and probability-based metrics.

## Work Completed

- Loaded the trained CTR model.
- Predicted click probabilities.
- Calculated AUC-ROC.
- Calculated Log Loss.
- Computed CTR@10.
- Saved evaluation results.

## Files Generated

- ctr_evaluation.csv

## Outcome

The CTR prediction model was evaluated using recommendation-oriented metrics. The model is now ready to provide click probabilities that will be combined with Collaborative Filtering scores in Stage 3.